In [1]:
import numpy as np
import tvm
from tvm.script import tir as T


[08:54:55] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:55] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:55] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m2` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:55] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m2` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:55] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:55] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: E

## Element-wise Add


In [2]:
# init data
a = np.arange(16).reshape(4, 4)
b = np.arange(16, 0, -1).reshape(4, 4)

In [3]:
c_np = a + b
c_np

array([[16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16]])

In [4]:
# low-level numpy version
def lnumpy_add(a: np.ndarray, b: np.ndarray, c: np.ndarray):
    for i in range(4):
        for j in range(4):
            c[i, j] = a[i, j] + b[i, j]


c_lnumpy = np.empty((4, 4), dtype=np.int64)
lnumpy_add(a, b, c_lnumpy)
c_lnumpy


array([[16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16]])

In [5]:
# TensorIR version
@tvm.script.ir_module
class MyAdd:
    @T.prim_func
    def add(
        A: T.Buffer((4, 4), "int64"),
        B: T.Buffer((4, 4), "int64"),
        C: T.Buffer((4, 4), "int64"),
    ):
        T.func_attr({"global_symbol": "add"})
        for i, j in T.grid(4, 4):
            with T.sblock("C"):
                vi = T.axis.spatial(4, i)
                vj = T.axis.spatial(4, j)
                C[vi, vj] = A[vi, vj] + B[vi, vj]


rt_lib = tvm.compile(MyAdd, target="llvm")
a_tvm = tvm.runtime.tensor(a)
b_tvm = tvm.runtime.tensor(b)
c_tvm = tvm.runtime.tensor(np.empty((4, 4), dtype=np.int64))
rt_lib["add"](a_tvm, b_tvm, c_tvm)
np.testing.assert_allclose(c_tvm.numpy(), c_np, rtol=1e-5)


# Broadcast Add


In [6]:
# init data
a = np.arange(16).reshape(4, 4)
b = np.arange(4, 0, -1).reshape(4)

In [7]:
# numpy version
c_np = a + b
c_np

array([[ 4,  4,  4,  4],
       [ 8,  8,  8,  8],
       [12, 12, 12, 12],
       [16, 16, 16, 16]])

In [8]:
@tvm.script.ir_module
class MyAdd:
    @T.prim_func
    def add(
        A: T.Buffer((4, 4), "int64"),
        B: T.Buffer((4), "int64"),
        C: T.Buffer((4, 4), "int64"),
    ):
        T.func_attr({"global_symbol": "add", "tir.noalias": True})
        for i, j in T.grid(4, 4):
            with T.sblock("C"):
                vi = T.axis.spatial(4, i)
                vj = T.axis.spatial(4, j)
                C[vi, vj] = A[vi, vj] + B[vj]


rt_lib = tvm.compile(MyAdd, target="llvm")
a_tvm = tvm.runtime.tensor(a)
b_tvm = tvm.runtime.tensor(b)
c_tvm = tvm.runtime.tensor(np.empty((4, 4), dtype=np.int64))
rt_lib["add"](a_tvm, b_tvm, c_tvm)
np.testing.assert_allclose(c_tvm.numpy(), c_np, rtol=1e-5)

## 2D Convolution


In [9]:
N, CI, H, W, CO, K = 1, 1, 8, 8, 2, 3
OUT_H, OUT_W = H - K + 1, W - K + 1
data = np.arange(N * CI * H * W).reshape(N, CI, H, W)
weight = np.arange(CO * CI * K * K).reshape(CO, CI, K, K)

In [10]:
# torch version
import torch

data_torch = torch.Tensor(data)
weight_torch = torch.Tensor(weight)
conv_torch = torch.nn.functional.conv2d(data_torch, weight_torch)
conv_torch = conv_torch.numpy().astype(np.int64)
conv_torch

array([[[[ 474,  510,  546,  582,  618,  654],
         [ 762,  798,  834,  870,  906,  942],
         [1050, 1086, 1122, 1158, 1194, 1230],
         [1338, 1374, 1410, 1446, 1482, 1518],
         [1626, 1662, 1698, 1734, 1770, 1806],
         [1914, 1950, 1986, 2022, 2058, 2094]],

        [[1203, 1320, 1437, 1554, 1671, 1788],
         [2139, 2256, 2373, 2490, 2607, 2724],
         [3075, 3192, 3309, 3426, 3543, 3660],
         [4011, 4128, 4245, 4362, 4479, 4596],
         [4947, 5064, 5181, 5298, 5415, 5532],
         [5883, 6000, 6117, 6234, 6351, 6468]]]])

In [11]:
conv_torch.shape

(1, 2, 6, 6)

In [12]:
data.shape, weight.shape

((1, 1, 8, 8), (2, 1, 3, 3))

In [ ]:
@tvm.script.ir_module
class MyConv:
    @T.prim_func
    def conv(
        data: T.Buffer((1, 1, 8, 8), "int64"),
        weight: T.Buffer((2, 1, 3, 3), "int64"),
        conv: T.Buffer((1, 2, 6, 6), "int64"),
    ):
        T.func_attr({"global_symbol": "conv", "tirx.noalias": True})
        for n, co, h, w in T.grid(1, 2, 6, 6):
            with T.sblock("conv"):
                vn = T.axis.spatial(1, n)
                vco = T.axis.spatial(2, co)
                vh = T.axis.spatial(6, h)
                vw = T.axis.spatial(6, w)
                conv[vn, vco, vh, vw] = 0
                for ci in range(1):
                    for kh in range(3):
                        for kw in range(3):
                            conv[vn, vco, vh, vw] += (
                                data[vn, ci, vh + kh, vw + kw] * weight[vco, ci, kh, kw]
                            )

    @T.prim_func
    def conv_opt(
        data: T.Buffer((1, 1, 8, 8), "int64"),
        weight: T.Buffer((2, 1, 3, 3), "int64"),
        conv: T.Buffer((1, 2, 6, 6), "int64"),
    ):
        T.func_attr({"global_symbol": "conv_opt", "tirx.noalias": True})
        for n, co, h, w, ci, kh, kw in T.grid(1, 2, 6, 6, 1, 3, 3):
            with T.sblock("conv"):
                vn = T.axis.spatial(1, n)
                vco = T.axis.spatial(2, co)
                vh = T.axis.spatial(6, h)
                vw = T.axis.spatial(6, w)
                vci = T.axis.reduce(1, ci)
                vkh = T.axis.reduce(3, kh)
                vkw = T.axis.reduce(3, kw)
                with T.init():
                    conv[vn, vco, vh, vw] = 0
                conv[vn, vco, vh, vw] += (
                    data[vn, vci, vh + vkh, vw + vkw] * weight[vco, vci, vkh, vkw]
                )


rt_lib = tvm.compile(MyConv, target="llvm")
data_tvm = tvm.runtime.tensor(data)
weight_tvm = tvm.runtime.tensor(weight)
conv_tvm = tvm.runtime.tensor(np.empty((N, CO, OUT_H, OUT_W), dtype=np.int64))
rt_lib["conv"](data_tvm, weight_tvm, conv_tvm)
np.testing.assert_allclose(conv_tvm.numpy(), conv_torch, rtol=1e-5)

rt_lib["conv_opt"](data_tvm, weight_tvm, conv_tvm)
np.testing.assert_allclose(conv_tvm.numpy(), conv_torch, rtol=1e-5)

ex_conv = rt_lib.mod.time_evaluator("conv", tvm.cpu(0), number=100)
ex_conv_opt = rt_lib.mod.time_evaluator("conv_opt", tvm.cpu(0), number=100)
print("conv:", ex_conv(data_tvm, weight_tvm, conv_tvm).mean * 1e3, "ms")
print("conv_opt:", ex_conv_opt(data_tvm, weight_tvm, conv_tvm).mean * 1e3, "ms")

conv: 0.00024896 ms
conv_opt: 0.00024936 ms
